# Lesson 2 — Tool Use

**Goal:** Give your agent Python functions it can call. This is what makes it truly agentic.

📖 Follow the step-by-step guide: [lessons/0002-tool-use.html](../lessons/0002-tool-use.html)

---

### The agent loop

```
1. Send message + tools → Claude
2. stop_reason == "end_turn"  → Done. Return text.
   stop_reason == "tool_use" → Execute tool(s) → send results back → go to 2
```

> ☕ **Java analogy:** Tool use is like callback interfaces — you define the contract,
> Claude decides when to invoke it, your code executes it.

---

## Cell 1 — Setup

In [ ]:
import anthropic
import datetime

client = anthropic.Anthropic()
print("Client ready!")

## Cell 2 — Define a tool function and its schema

Two parts for every tool:
1. **The Python function** — what your code actually runs
2. **The JSON schema** — what Claude reads to know the tool exists and how to call it

In [ ]:
# ── 1. The actual Python function ──────────────────────────
def get_current_date():
    """Return today's date as an ISO string."""
    return datetime.date.today().isoformat()  # e.g. "2026-06-21"


# ── 2. The tool schema Claude reads ───────────────────────
tools = [
    {
        "name": "get_current_date",
        "description": "Returns today's date in ISO format (YYYY-MM-DD).",
        "input_schema": {
            "type": "object",
            "properties": {},   # no parameters
            "required": []
        }
    }
]

# Quick sanity check
print("Today:", get_current_date())

## Cell 3 — The agent loop (minimal version)

In [ ]:
def run_agent(user_message):
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )

        # Finished — return the text
        if response.stop_reason == "end_turn":
            return response.content[0].text

        # Claude wants to use a tool
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  → Calling: {block.name}({block.input})")

                    if block.name == "get_current_date":
                        result = get_current_date()
                    else:
                        result = f"Unknown tool: {block.name}"

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })

            messages.append({"role": "user", "content": tool_results})


# Test it!
answer = run_agent("What day of the week is today? Use the tool to get the date.")
print("\nAgent:", answer)

## Cell 4 — A tool with parameters (calculator)

In [ ]:
def calculate(expression: str) -> str:
    """Safely evaluate a simple arithmetic expression."""
    allowed = set("0123456789+-*/().,% ")
    if not all(c in allowed for c in expression):
        return "Error: only arithmetic expressions allowed"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


calc_tool = {
    "name": "calculate",
    "description": "Evaluate a basic arithmetic expression and return the numeric result.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The arithmetic expression to evaluate, e.g. '42 * 1.15'"
            }
        },
        "required": ["expression"]
    }
}

# Quick test
print(calculate("48.50 * 0.15"))  # should print 7.275

## Cell 5 — Multi-tool agent with a dispatch function

A clean pattern: one `dispatch_tool` function routes tool names to Python functions.

> ☕ **Java analogy:** This is the Strategy pattern — a `Map<String, Function>` dispatcher.

In [ ]:
all_tools = [tools[0], calc_tool]  # date + calculator


def dispatch_tool(name, inputs):
    """Route tool name → Python function."""
    if name == "get_current_date":
        return get_current_date()
    if name == "calculate":
        return calculate(inputs["expression"])
    return f"Unknown tool: {name}"


def run_agent_v2(user_message, available_tools):
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=available_tools,
            messages=messages
        )
        if response.stop_reason == "end_turn":
            return response.content[0].text

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = dispatch_tool(block.name, block.input)
                print(f"  Tool: {block.name}({block.input}) → {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result)
                })
        messages.append({"role": "user", "content": tool_results})


# Test with a question that needs BOTH tools
print(run_agent_v2(
    "What is 15% tip on a $48.50 dinner bill? Also tell me today's date.",
    all_tools
))

## Cell 6 — Experiment yourself 🧪

Try adding a new tool: `get_temperature(city: str)` that returns a fake temperature.
Wire it into `dispatch_tool` and test with a message like:
*"What's the temperature in Singapore and what is 32 degrees Celsius in Fahrenheit?"*

In [ ]:
# Your code here!
def get_temperature(city: str) -> str:
    # TODO: return a fake temperature for any city
    pass


temp_tool = {
    "name": "get_temperature",
    "description": "...",  # TODO: fill in
    "input_schema": {
        # TODO: fill in
    }
}

---

## ✅ Lesson 2 Complete

You can now:
- Define a tool with a JSON schema
- Write the agent loop that handles `stop_reason == "tool_use"`
- Dispatch tool calls to Python functions
- Wire multiple tools into one agent

**Next:** [Lesson 3 — The Agent Loop](lesson-03-agent-loop.ipynb) — deeper patterns for robust looping.

💬 **Ask your teacher** in the Claude Code chat if you get stuck.